In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "GraphLP"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# ========= 2) 参数模板 =========
params = {
    "c": np.array([1.0, 2.0, 3.0]),           # 路径弧代价
    "A_ub": np.array([[1, 1, 0], [0, 1, 1]]), # 约束矩阵
    "b_ub": np.array([1, 1]),
    "bounds": [(0,1), (0,1), (0,1)]           # 0-1或连续边界
}
res = linprog(params["c"], A_ub=params["A_ub"], b_ub=params["b_ub"], bounds=params["bounds"], method="highs")
print(res.x, res.fun)


In [ ]:
"""
图 + 规划模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "图 + 规划模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
FROM_COLUMN = "起点"  # TODO: 请填写[起点列名]，说明：边的起始节点。
TO_COLUMN = "终点"  # TODO: 请填写[终点列名]，说明：边的结束节点。
COST_COLUMN = "成本"  # TODO: 请填写[单位成本列名]，说明：每单位流量成本。
CAPACITY_COLUMN = "容量"  # TODO: 请填写[容量列名]，说明：边流量上限。
DEMAND_DICT = {"源点": -10, "汇点": 10}  # TODO: 请填写[节点供需字典]，说明：供给为负、需求为正，所有节点供需和应为 0。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # data 每行表示一条边：起点、终点、单位成本、容量。
    nodes = sorted(set(data[FROM_COLUMN]) | set(data[TO_COLUMN]))
    edges = list(data[[FROM_COLUMN, TO_COLUMN]].itertuples(index=False, name=None))
    c = data[COST_COLUMN].to_numpy(dtype=float)
    bounds = [(0, cap) for cap in data[CAPACITY_COLUMN].to_numpy(dtype=float)]
    A_eq, b_eq = [], []
    for node in nodes:
        row = []
        for u, v in edges:
            row.append((1 if v == node else 0) - (1 if u == node else 0))
        A_eq.append(row)
        b_eq.append(DEMAND_DICT.get(node, 0))
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    result = data.copy()
    result["最优流量"] = res.x if res.success else np.nan
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(res.message)
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
